# Fine-tune E5-base — `train_5000` / `valid_5000` / `test_5000` (1 epoch)

Fine-tune **`intfloat/multilingual-e5-base`** cho bài toán truy hồi sản phẩm tiếng Việt với:

| File | Mô tả |
|---|---|
| `data/training/train_5000.jsonl` | 3500 cặp train |
| `data/training/valid_5000.jsonl` | 750 cặp validation |
| `data/training/test_5000.jsonl` | 750 cặp test |

**Loss:** `MultipleNegativesRankingLoss` (positive pair + in-batch negatives).

**Output:** `embedding_project/models/e5_base_finetuned_final/`

> Chạy trên **GPU** (Colab T4/V100). ~3500 mẫu × 1 epoch ≈ 15–30 phút trên T4.

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" "accelerate>=1.1.0" torch datasets pandas scikit-learn numpy tqdm

## 2) Clone repo (Colab) hoặc dùng thư mục local

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"
COLAB_REPO = Path("/content/llm_provider_benchmarking")
LOCAL_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("c:/llm_provider_benchmarking"),
]

def find_local_repo() -> Path | None:
    for root in LOCAL_CANDIDATES:
        if (root / "data" / "training" / "train_5000.jsonl").is_file():
            return root.resolve()
    return None

if COLAB_REPO.is_dir() and (COLAB_REPO / ".git").exists():
    REPO_DIR = COLAB_REPO
elif find_local_repo() is not None:
    REPO_DIR = find_local_repo()
    print("Dùng repo local:", REPO_DIR)
else:
    if COLAB_REPO.exists():
        shutil.rmtree(COLAB_REPO)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO)], check=True)
    REPO_DIR = COLAB_REPO
    print("Cloned:", REPO_URL)

SCRIPTS_DIR = REPO_DIR / "embedding_project" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

PROJECT_ROOT = REPO_DIR / "embedding_project"
DATA_DIR = REPO_DIR / "data" / "training"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

print("REPO_DIR:", REPO_DIR)
print("DATA_DIR:", DATA_DIR)

## 3) Kiểm tra GPU và dữ liệu

In [ ]:
import torch

TRAIN_PATH = DATA_DIR / "train_5000.jsonl"
VALID_PATH = DATA_DIR / "valid_5000.jsonl"
TEST_PATH = DATA_DIR / "test_5000.jsonl"

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for label, path in [
    ("train", TRAIN_PATH),
    ("valid", VALID_PATH),
    ("test", TEST_PATH),
]:
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu file {label}: {path}")
    n = sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.strip())
    print(f"{label:5} {n:5} rows  {path.name}")

## 4) Cấu hình training (1 epoch)

In [ ]:
from model_presets import get_preset

PRESET = get_preset("e5-base")

USE_GPU = torch.cuda.is_available()
EPOCHS = 1
BATCH_SIZE = 8 if USE_GPU else 2
FP16 = USE_GPU
MAX_SEQ_LENGTH = PRESET.max_seq_length
FINAL_DIR = MODELS_DIR / PRESET.final_subdir

print("Base model:", PRESET.base_model)
print("Output:", FINAL_DIR)
print(
    f"epochs={EPOCHS} batch={BATCH_SIZE} lr={PRESET.learning_rate} "
    f"warmup={PRESET.warmup_ratio} fp16={FP16} max_seq={MAX_SEQ_LENGTH}"
)

## 5) Load train / valid

Mỗi dòng JSON có `query`, `positive` (và tùy chọn `negative`, `hard_negative`). Fine-tune dùng cặp `query` → `positive`.

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def to_pair_dataset(rows: list[dict]) -> Dataset:
    pairs = [
        {"anchor": r["query"], "positive": r["positive"]}
        for r in rows
        if r.get("query") and r.get("positive")
    ]
    if not pairs:
        raise ValueError("Không có cặp query/positive hợp lệ")
    return Dataset.from_list(pairs)

train_rows = load_jsonl(TRAIN_PATH)
valid_rows = load_jsonl(VALID_PATH)
test_rows = load_jsonl(TEST_PATH)

train_ds = to_pair_dataset(train_rows)
valid_ds = to_pair_dataset(valid_rows)

print("train pairs:", len(train_ds))
print("valid pairs:", len(valid_ds))
print("test rows:", len(test_rows))
print("sample query:", train_rows[0]["query"][:100], "...")

## 6) Fine-tune E5-base (1 epoch)

Khi **inference**, E5 cần prefix `query:` / `passage:` — xem cell đánh giá bên dưới.

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers, SentenceTransformerTrainingArguments

model = SentenceTransformer(PRESET.base_model)
model.max_seq_length = MAX_SEQ_LENGTH
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(MODELS_DIR / PRESET.name),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=PRESET.learning_rate,
    warmup_ratio=PRESET.warmup_ratio,
    fp16=FP16,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=20,
    save_total_limit=2,
    run_name="e5-base-train5000-1epoch",
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=loss,
)

trainer.train()
model.save(str(FINAL_DIR))
print("Saved model ->", FINAL_DIR)

## 7) Đánh giá trên test set

Với mỗi query, xếp hạng `positive` cùng các `hard_negative` (hoặc `negative` nếu không có hard negative).
Báo cáo **Accuracy@1** (positive đứng đầu) và **MRR**.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

def normalize(x: np.ndarray) -> np.ndarray:
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-12)

def e5_query(text: str) -> str:
    return f"query: {text}"

def e5_passage(text: str) -> str:
    return f"passage: {text}"

def negatives_for_row(row: dict) -> list[str]:
    hard = row.get("hard_negative") or []
    if hard:
        return list(hard)
    return list(row.get("negative") or [])

def evaluate_pairwise_ranking(model_path: str, rows: list[dict], batch_size: int = 32) -> dict:
    m = SentenceTransformer(model_path)
    m.max_seq_length = MAX_SEQ_LENGTH

    acc1_list, mrr_list = [], []
    skipped = 0

    for row in rows:
        query = row.get("query")
        positive = row.get("positive")
        negs = negatives_for_row(row)
        if not query or not positive or not negs:
            skipped += 1
            continue

        candidates = [positive] + negs
        q_emb = normalize(m.encode([e5_query(query)], batch_size=1, convert_to_numpy=True))
        c_emb = normalize(
            m.encode(
                [e5_passage(t) for t in candidates],
                batch_size=batch_size,
                convert_to_numpy=True,
                show_progress_bar=False,
            )
        )
        scores = (q_emb @ c_emb.T).ravel()
        rank = int(np.argsort(-scores).tolist().index(0)) + 1  # positive ở index 0
        acc1_list.append(1.0 if rank == 1 else 0.0)
        mrr_list.append(1.0 / rank)

    return {
        "n_evaluated": len(acc1_list),
        "n_skipped": skipped,
        "Accuracy@1": float(np.mean(acc1_list)) if acc1_list else 0.0,
        "MRR": float(np.mean(mrr_list)) if mrr_list else 0.0,
    }

pretrained_metrics = evaluate_pairwise_ranking(PRESET.base_model, test_rows)
finetuned_metrics = evaluate_pairwise_ranking(str(FINAL_DIR), test_rows)

result = {
    "dataset": "test_5000.jsonl",
    "epochs": EPOCHS,
    "pretrained": pretrained_metrics,
    "finetuned": finetuned_metrics,
}

import json
print(json.dumps(result, ensure_ascii=False, indent=2))

metrics_path = OUTPUT_EVAL_DIR / "metrics_e5_base_train5000_1epoch.json"
metrics_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved metrics ->", metrics_path)

## 8) Lưu model (Colab: Drive + tải zip)

In [ ]:
import shutil
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_name = f"e5_base_finetuned_final_{stamp}.zip"
zip_base = f"/content/{zip_name.replace('.zip', '')}"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=FINAL_DIR)
print("Zip:", zip_path, f"({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

try:
    from google.colab import drive, files

    drive.mount("/content/drive", force_remount=False)
    drive_out = Path("/content/drive/MyDrive/models") / zip_name
    drive_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, drive_out)
    print("Saved to Drive:", drive_out)
    files.download(zip_path)
except ImportError:
    print("Không phải Colab — bỏ qua Drive/download. Model đã lưu tại:", FINAL_DIR)